In [37]:
import pandas as pd

In [38]:
def derive_severity(score, version):
    if score is None:
        return None

    if version and str(version).startswith("3"):
        # CVSS v3.x scale
        if score == 0.0:
            return "NONE"
        elif score <= 3.9:
            return "LOW"
        elif score <= 6.9:
            return "MEDIUM"
        elif score <= 8.9:
            return "HIGH"
        else:
            return "CRITICAL"
    else:
        # CVSS v2 scale
        if score <= 3.9:
            return "LOW"
        elif score <= 6.9:
            return "MEDIUM"
        else:
            return "HIGH"


In [39]:
def parse_cve(cve_dict):
    cve = cve_dict["cve"]  # already inside the "cve" object

    # Basic info
    cve_id = cve.get("id")
    published = cve.get("published")
    last_modified = cve.get("lastModified")
    status = cve.get("vulnStatus")

    # English description
    description = next(
        (d["value"] for d in cve.get("descriptions", []) if d.get("lang") == "en"),
        None,
    )

    # CVSS (prefer v3.1 > v3.0 > v2)
    cvss_score, cvss_vector, cvss_version, severity = None, None, None, None
    metrics = cve.get("metrics", {})

    if "cvssMetricV31" in metrics:
        metric = metrics["cvssMetricV31"][0]
        cvss_score = metric["cvssData"]["baseScore"]
        cvss_vector = metric["cvssData"]["vectorString"]
        cvss_version = metric["cvssData"]["version"]
        severity = metric.get("baseSeverity")

    elif "cvssMetricV30" in metrics:
        metric = metrics["cvssMetricV30"][0]
        cvss_score = metric["cvssData"]["baseScore"]
        cvss_vector = metric["cvssData"]["vectorString"]
        cvss_version = metric["cvssData"]["version"]
        severity = metric.get("baseSeverity")

    elif "cvssMetricV2" in metrics:
        metric = metrics["cvssMetricV2"][0]
        cvss_score = metric["cvssData"]["baseScore"]
        cvss_vector = metric["cvssData"]["vectorString"]
        cvss_version = metric["cvssData"]["version"]
        severity = metric.get("baseSeverity")

    # ✅ Always apply fallback (both v2 and v3)
    if not severity and cvss_score is not None:
        severity = derive_severity(cvss_score, cvss_version)

    # Weakness (CWE) – allow multiple
    cwes = []
    for w in cve.get("weaknesses", []):
        for d in w.get("description", []):
            if d.get("lang") == "en":
                cwes.append(d["value"])
    cwe = "; ".join(cwes) if cwes else None

    # Affected products (CPEs) → split vendor/product
    cpes, vendors, products = [], [], []
    for config in cve.get("configurations", []):
        for node in config.get("nodes", []):
            for match in node.get("cpeMatch", []):
                if match.get("vulnerable"):
                    cpe_uri = match["criteria"]
                    cpes.append(cpe_uri)

                    parts = cpe_uri.split(":")
                    if len(parts) >= 5:
                        vendors.append(parts[3])
                        products.append(parts[4])

    # References
    refs = [ref["url"] for ref in cve.get("references", [])]

    # CVE Tags
    tags = []
    for tag in cve.get("cveTags", []):
        if isinstance(tag, dict) and "tag" in tag:
            tags.append(tag["tag"])
        elif isinstance(tag, str):
            tags.append(tag)
    tags = "; ".join(tags) if tags else None

    return {
        "CVE_ID": cve_id,
        "Published": published,
        "Last_Modified": last_modified,
        "Status": status,
        "Description": description,
        "CVSS_BaseScore": cvss_score,
        "CVSS_Vector": cvss_vector,
        "CVSS_Version": cvss_version,
        "CVSS_Severity": severity,
        "Weakness_CWE": cwe,
        "Affected_CPEs": "; ".join(cpes) if cpes else None,
        "Vendor": "; ".join(set(vendors)) if vendors else None,
        "Product": "; ".join(set(products)) if products else None,
        "References": "; ".join(refs) if refs else None,
        "CVE_Tags": tags,
    }

In [40]:
year = [2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]

records = []

for i in year:
    df = pd.read_json(fr'JSON-FEED\nvdcve-2.0-{i}.json')

    cve_list  = df['vulnerabilities']

    for item in cve_list:
        records.append(parse_cve(item))

In [41]:
parsed_json_feed = pd.DataFrame(records)

In [42]:
parsed_json_feed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 242284 entries, 0 to 242283
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   CVE_ID          242284 non-null  object 
 1   Published       242284 non-null  object 
 2   Last_Modified   242284 non-null  object 
 3   Status          242284 non-null  object 
 4   Description     242284 non-null  object 
 5   CVSS_BaseScore  223939 non-null  float64
 6   CVSS_Vector     223939 non-null  object 
 7   CVSS_Version    223939 non-null  object 
 8   CVSS_Severity   223939 non-null  object 
 9   Weakness_CWE    223738 non-null  object 
 10  Affected_CPEs   203022 non-null  object 
 11  Vendor          203022 non-null  object 
 12  Product         203022 non-null  object 
 13  References      228937 non-null  object 
 14  CVE_Tags        0 non-null       object 
dtypes: float64(1), object(14)
memory usage: 27.7+ MB


In [43]:
parsed_json_feed.drop('CVE_Tags',axis=1,inplace=True)

In [44]:
parsed_json_feed[:10]

,CVE_ID,Published,Last_Modified,Status,Description,CVSS_BaseScore,CVSS_Vector,CVSS_Version,CVSS_Severity,Weakness_CWE,Affected_CPEs,Vendor,Product,References
0,CVE-2014-0791,2014-01-03T18:54:13.257,2025-04-11T00:51:21.963,Deferred,Integer overflow in the license_read_scope_lis...,6.8,AV:N/AC:M/Au:N/C:P/I:P/A:P,2.0,MEDIUM,CWE-189,cpe:2.3:a:freerdp:freerdp:1.0.0:*:*:*:*:*:*:*;...,freerdp,freerdp,http://advisories.mageia.org/MGASA-2014-0287.h...
1,CVE-2014-0620,2014-01-08T15:30:02.683,2025-04-11T00:51:21.963,Deferred,Multiple cross-site scripting (XSS) vulnerabil...,4.3,AV:N/AC:M/Au:N/C:N/I:P/A:N,2.0,MEDIUM,CWE-79,cpe:2.3:o:technicolor:tc7200_firmware:std6.01....,technicolor,tc7200; tc7200_firmware,http://www.exploit-db.com/exploits/30668; http...
2,CVE-2014-0621,2014-01-08T15:30:02.730,2025-04-11T00:51:21.963,Deferred,Multiple cross-site request forgery (CSRF) vul...,6.8,AV:N/AC:M/Au:N/C:P/I:P/A:P,2.0,MEDIUM,CWE-352,cpe:2.3:o:technicolor:tc7200_firmware:std6.01....,technicolor,tc7200; tc7200_firmware,http://www.exploit-db.com/exploits/30667; http...
3,CVE-2014-1232,2014-01-08T15:30:02.747,2025-04-11T00:51:21.963,Deferred,Cross-site scripting (XSS) vulnerability in th...,4.3,AV:N/AC:M/Au:N/C:N/I:P/A:N,2.0,MEDIUM,CWE-79,cpe:2.3:a:foliovision:foliopress_wysiwyg:*:*:*...,foliovision,foliopress_wysiwyg,http://secunia.com/advisories/56261; http://wo...
4,CVE-2014-0651,2014-01-08T21:55:06.223,2025-04-11T00:51:21.963,Deferred,The administrative interface in Cisco Context ...,4.9,AV:N/AC:M/Au:S/C:P/I:P/A:N,2.0,MEDIUM,CWE-264,cpe:2.3:a:cisco:context_directory_agent:-:*:*:...,cisco,context_directory_agent,http://osvdb.org/101809; http://secunia.com/ad...
5,CVE-2014-0652,2014-01-08T21:55:06.240,2025-04-11T00:51:21.963,Deferred,Cross-site scripting (XSS) vulnerability in th...,4.3,AV:N/AC:M/Au:N/C:N/I:P/A:N,2.0,MEDIUM,CWE-79,cpe:2.3:a:cisco:context_directory_agent:-:*:*:...,cisco,context_directory_agent,http://osvdb.org/101803; http://secunia.com/ad...
6,CVE-2014-0653,2014-01-08T21:55:06.270,2025-04-11T00:51:21.963,Deferred,The Identity Firewall (IDFW) functionality in ...,4.3,AV:N/AC:M/Au:N/C:N/I:P/A:N,2.0,MEDIUM,CWE-20,cpe:2.3:h:cisco:adaptive_security_appliance:*:...,cisco,adaptive_security_appliance,http://osvdb.org/101834; http://secunia.com/ad...
7,CVE-2014-0654,2014-01-08T21:55:06.303,2025-04-11T00:51:21.963,Deferred,Cisco Context Directory Agent (CDA) allows rem...,4.3,AV:N/AC:M/Au:N/C:N/I:P/A:N,2.0,MEDIUM,CWE-20,cpe:2.3:a:cisco:context_directory_agent:-:*:*:...,cisco,context_directory_agent,http://osvdb.org/101802; http://secunia.com/ad...
8,CVE-2014-0655,2014-01-08T21:55:06.333,2025-04-11T00:51:21.963,Deferred,The Identity Firewall (IDFW) functionality in ...,4.3,AV:N/AC:M/Au:N/C:N/I:P/A:N,2.0,MEDIUM,CWE-20,cpe:2.3:h:cisco:adaptive_security_appliance:*:...,cisco,adaptive_security_appliance,http://osvdb.org/101838; http://secunia.com/ad...
9,CVE-2014-0656,2014-01-08T21:55:06.380,2025-04-11T00:51:21.963,Deferred,Cisco Context Directory Agent (CDA) allows rem...,4.0,AV:N/AC:L/Au:S/C:N/I:P/A:N,2.0,MEDIUM,CWE-20,cpe:2.3:a:cisco:context_directory_agent:-:*:*:...,cisco,context_directory_agent,http://osvdb.org/101801; http://tools.cisco.co...


In [45]:
parsed_json_feed.to_csv("CVE_2014_to_2025.csv",index=False, encoding="utf-8")

In [1]:
import pandas as pd

df_test = pd.read_csv("CVE_2014_to_2025.csv")

In [2]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 242284 entries, 0 to 242283
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   CVE_ID          242284 non-null  object 
 1   Published       242284 non-null  object 
 2   Last_Modified   242284 non-null  object 
 3   Status          242284 non-null  object 
 4   Description     242284 non-null  object 
 5   CVSS_BaseScore  223939 non-null  float64
 6   CVSS_Vector     223939 non-null  object 
 7   CVSS_Version    223939 non-null  float64
 8   CVSS_Severity   223939 non-null  object 
 9   Weakness_CWE    223738 non-null  object 
 10  Affected_CPEs   203022 non-null  object 
 11  Vendor          203022 non-null  object 
 12  Product         203022 non-null  object 
 13  References      228937 non-null  object 
dtypes: float64(2), object(12)
memory usage: 25.9+ MB


In [4]:
df_test.dropna(inplace=True)

In [1]:
df_test

NameError: name 'df_test' is not defined

In [8]:
df_test.to_csv("CVE_2014_to_2025v2.csv",index=False, encoding="utf-8")